<a href="https://colab.research.google.com/github/fahadelahikhan/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%%bash
git clone https://github.com/fahadelahikhan/flyrank-ml-internship.git
cp -r flyrank-ml-internship/* .
ls -la

total 104
drwxr-xr-x  1 root root  4096 Oct 29 18:31 .
drwxr-xr-x  1 root root  4096 Oct 29 18:28 ..
-rw-r--r--  1 root root   654 Oct 29 18:31 AGENTS.md
-rw-r--r--  1 root root   654 Oct 29 18:31 CLAUDE.md
drwxr-xr-x  4 root root  4096 Oct 21 13:28 .config
drwxr-xr-x  3 root root  4096 Oct 29 18:31 data
-rw-r--r--  1 root root  2763 Oct 29 18:31 DATA_USE.md
drwxr-xr-x  3 root root  4096 Oct 29 18:31 docs
drwxr-xr-x 12 root root  4096 Oct 29 18:31 flyrank-ml-internship
-rw-r--r--  1 root root 10408 Oct 29 18:31 GUIDE.md
-rw-r--r--  1 root root  1289 Oct 29 18:31 LICENSE
drwxr-xr-x  2 root root  4096 Oct 29 18:31 notebooks
drwxr-xr-x  3 root root  4096 Oct 29 18:31 outputs
-rw-r--r--  1 root root  9821 Oct 29 18:31 README.md
-rw-r--r--  1 root root   107 Oct 29 18:31 requirements.txt
drwxr-xr-x  1 root root  4096 Oct 21 13:28 sample_data
drwxr-xr-x  2 root root  4096 Oct 29 18:31 scripts
-rw-r--r--  1 root root  5671 Oct 29 18:31 SETUP.md
drwxr-xr-x 14 root root  4096 Oct 29 18:31 skill

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [9]:
import os

def print_file_if_exists(path):
    if os.path.exists(path):
        print(f"=== {path} ===")
        with open(path, 'r') as f:
            print(f.read())
    else:
        print(f"{path} does not exist.")

print_file_if_exists('skills/writing-data-contracts/SKILL.md')
print_file_if_exists('skills/flyrank/flyrank-data/SKILL.md')

=== skills/writing-data-contracts/SKILL.md ===
---
name: writing-data-contracts
description: Writes a data contract — what a row means, which fields are features vs labels vs context vs excluded, over which time windows — and verifies every claim with a query. Use before any feature building or modeling, or when results look wrong and the data definition is suspect.
---

# Writing data contracts

Most weak analysis fails here, before any model: nobody wrote down what a row means. A data
contract is that write-down — and a contract your code has checked, not just stated.

## The contract, section by section

1. **Unit of analysis.** One row = one WHAT? (a page? a page-day? a client?) State it.
2. **Time window.** Which dates does each field cover? Draw the windows if they differ.
3. **Field classification.** Every field you may touch goes in exactly one bucket:
   - **Feature** — knowable BEFORE the moment you predict, safe to use.
   - **Label / proxy** — the thing you predict, or what

In [10]:
import os

def print_file_if_exists(path):
    if os.path.exists(path):
        print(f"=== {path} ===")
        with open(path, 'r') as f:
            print(f.read())
    else:
        print(f"{path} does not exist.")

print_file_if_exists('skills/writing-data-contracts/SKILL.md')
print_file_if_exists('skills/flyrank/flyrank-data/SKILL.md')

=== skills/writing-data-contracts/SKILL.md ===
---
name: writing-data-contracts
description: Writes a data contract — what a row means, which fields are features vs labels vs context vs excluded, over which time windows — and verifies every claim with a query. Use before any feature building or modeling, or when results look wrong and the data definition is suspect.
---

# Writing data contracts

Most weak analysis fails here, before any model: nobody wrote down what a row means. A data
contract is that write-down — and a contract your code has checked, not just stated.

## The contract, section by section

1. **Unit of analysis.** One row = one WHAT? (a page? a page-day? a client?) State it.
2. **Time window.** Which dates does each field cover? Draw the windows if they differ.
3. **Field classification.** Every field you may touch goes in exactly one bucket:
   - **Feature** — knowable BEFORE the moment you predict, safe to use.
   - **Label / proxy** — the thing you predict, or what

In [19]:
%%bash
pip install -q duckdb huggingface_hub

In [28]:
import os
import pandas as pd
import duckdb
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = token
except Exception as e:
    token = None

local_file = "/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet"

con = duckdb.connect()

if os.path.exists(local_file):
    print("--- Task 3 Verification on March 2026 Partition ---\n")

    # 1. Total Rows, Date Range
    q_stats = f"""
        SELECT
            COUNT(*) as total_rows,
            MIN(report_date) as min_date,
            MAX(report_date) as max_date
        FROM read_parquet('{local_file}')
    """
    print("Query 1: Total Rows & Date Bounds")
    print(con.execute(q_stats).df())

    # 2. Grain Probe (using correct hash columns)
    q_grain = f"""
        SELECT
            report_date, client_hash_id, content_hash_id, COUNT(*) as c
        FROM read_parquet('{local_file}')
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING c > 1
        LIMIT 5
    """
    print("\nQuery 2: Grain Probe (expecting empty result)")
    grain_df = con.execute(q_grain).df()
    if grain_df.empty:
        print("Pass: No duplicates found. The grain is unique.")
    else:
        print("Fail: Duplicates found:", grain_df)

    # 3. Availability Filter (where ga4_data_available = TRUE)
    q_avail = f"""
        SELECT
            COUNT(*) as active_ga4_rows,
            ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM read_parquet('{local_file}')), 2) as active_pct
        FROM read_parquet('{local_file}')
        WHERE ga4_data_available = TRUE
    """
    print("\nQuery 3: Active GA4 Availability Filter")
    print(con.execute(q_avail).df())
else:
    print(f"Local file not found at: {local_file}. Please check download steps.")

--- Task 3 Verification on March 2026 Partition ---

Query 1: Total Rows & Date Bounds
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

Query 2: Grain Probe (expecting empty result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pass: No duplicates found. The grain is unique.

Query 3: Active GA4 Availability Filter
   active_ga4_rows  active_pct
0           413966        4.21


In [11]:
with open('skills/flyrank/flyrank-data/SKILL.md', 'r') as f:
    print(f.read())

---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one ships in this repo; the big one is hosted and gated.

## 1. Starter dataset (in this repo)

`data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized
content item, 32 clients, trailing-90-day metrics. Full column reference: `docs/data-dictionary.md`
(keep it open). The gotchas that cause 90% of mistakes:

- **Rate columns are ×100 percentages**: `ctr = 0.76` means 0.76%, not 76%. Applies to ctr,
  engagement_rate, scroll_rate, ai_traffic_pct, trend_pct.
- **`avg_position = 0` means "no data"**, not rank zero (1,205 rows).
- **`scroll_rate` and `ai_traffic_pct` can e

In [12]:
# Let's print the rest of skills/flyrank/flyrank-data/SKILL.md if possible, or read its lines directly
with open('skills/flyrank/flyrank-data/SKILL.md', 'r') as f:
    lines = f.readlines()
    print("".join(lines[:100])) # Print the first 100 lines to see the details of dataset schema

---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one ships in this repo; the big one is hosted and gated.

## 1. Starter dataset (in this repo)

`data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized
content item, 32 clients, trailing-90-day metrics. Full column reference: `docs/data-dictionary.md`
(keep it open). The gotchas that cause 90% of mistakes:

- **Rate columns are ×100 percentages**: `ctr = 0.76` means 0.76%, not 76%. Applies to ctr,
  engagement_rate, scroll_rate, ai_traffic_pct, trend_pct.
- **`avg_position = 0` means "no data"**, not rank zero (1,205 rows).
- **`scroll_rate` and `ai_traffic_pct` can e

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadelahikhan/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Unit of Analysis
- **One Row** = One unique `content_id` (representing a specific pseudonymized content item/page belonging to a single pseudonymized client, indicated by `client_id`).
- Verification shows that grouping by `content_id` yields exactly 1 row per group (no duplicates). Therefore, the grain is perfectly unique at the `content_id` level.

### Time Window
- The aggregated metrics (such as impressions, clicks, pageviews, sessions, users, etc.) are computed over a **trailing 90-day window** (represented by the `*_90d` columns).
- The dataset also includes a split of this window into the **last 30 days** (`*_last_30d` columns) and the **previous 30 days** (`*_prev_30d` columns) to capture temporal trends.

In [15]:
import pandas as pd

# Load dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Claim 1: Unit of analysis is content_id (the grain)
duplicate_check = df.groupby('content_id').size().max()
print(f"Maximum rows per content_id (expecting 1): {duplicate_check}")
assert duplicate_check == 1, "Unit of analysis is not unique!"

# Claim 2: Shape and client counts
print(f"Total unique rows: {len(df)}")
print(f"Total unique clients: {df['client_id'].nunique()}")

Maximum rows per content_id (expecting 1): 1
Total unique rows: 30000
Total unique clients: 32


In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print("Shape of DataFrame:", df.shape)
print("Columns:", list(df.columns))
print("Total unique content_ids:", df['content_id'].nunique())
print("Total unique client_ids:", df['client_id'].nunique())

dups = df.groupby('content_id').size()
print("Duplicate rows based on content_id:", (dups > 1).sum())

Shape of DataFrame: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Total unique content_ids: 30000
Total unique client_ids: 32
Duplicate rows based on content_id: 0


In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print("Shape of DataFrame:", df.shape)
print("Columns:", list(df.columns))
print("Total unique content_ids:", df['content_id'].nunique())
print("Total unique client_ids:", df['client_id'].nunique())

# Let's probe if there are duplicate content_id rows
dups = df.groupby('content_id').size()
print("Duplicate rows based on content_id:", (dups > 1).sum())

Shape of DataFrame: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Total unique content_ids: 30000
Total unique client_ids: 32
Duplicate rows based on content_id: 0


## 2. Fields: feature / label / context / excluded

### Bucket Classification

- **Feature** (Safe to use, knowable BEFORE prediction):
  - Keywords & search metrics: `search_volume`, `competition`, `competition_level`, `cpc`.
  - Content characteristics: `content_type`, `main_intent`, `word_count`, `char_count`, `provider_used`, `model_used`, `word_count_tier`, `char_count_tier`.
  - Historical activity (safe windows): `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`.
  - Metadata & Age: `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`.

- **Label / Proxy** (The target we aim to predict or derived from):
  - Target label: `is_declining_label` (if created) or target metrics derived from `trend_direction` / `trend_pct`.
  - Proxy columns: `trend_direction`, `trend_pct`.

- **Context** (For grouping, joining, splitting, or reporting; never for the model to learn):
  - Identifiers: `content_id`, `client_id`.

- **Excluded** (Unsafe, future-leakage, or product-decision flags):
  - Overlapping window metrics: `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`.
  - *Why Excluded*: These either overlap with or contain future knowledge of the period we want to evaluate/forecast, introducing massive data leakage.

In [16]:
# Code validation of column sets to guarantee all columns are accounted for
all_cols = set(df.columns)
features = {'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'word_count_tier', 'char_count_tier', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier'}
labels = {'trend_direction', 'trend_pct'}
context = {'content_id', 'client_id'}
excluded = {
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'impression_tier', 'position_tier'
}

classified = features.union(labels).union(context).union(excluded)
missing_cols = all_cols - classified
print("Unclassified columns:", missing_cols)
assert len(missing_cols) == 0, f"Some columns were missed! {missing_cols}"

Unclassified columns: set()


## 3. Verify it with queries (grain, counts, missing values, windows)

### Verified Claims:
1. **Grain Check**: Grouping by `content_id` produces a maximum row count of 1, confirming the grain of the dataset is exactly one row per content item.
2. **Average Position Rule**: In GSC, `avg_position = 0` means 'no data' (no search impressions or rank), not rank 0. We verified that 1,205 rows have `avg_position == 0`.
3. **Rate Columns**: The columns `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, and `trend_pct` are represented as $\times100$ percentages (e.g. `ctr = 0.76` is $0.76\%$, not $76\%$).
4. **Missing Values**: Missingness is highly correlated with specific content types (e.g., `main_intent` and `content_type` determine which keyword metrics like CPC or competition level are missing).

In [17]:
# 1. Grain Probe Check
dups_count = (df.groupby('content_id').size() > 1).sum()
print(f"Number of duplicate content_id rows: {dups_count}")

# 2. Average Position '0' check
zero_positions = (df['avg_position'] == 0).sum()
print(f"Rows with avg_position == 0: {zero_positions}")

# 3. Missingness check
print("\nMissing values per column (ratio):")
missingness = df.isnull().mean()
print(missingness[missingness > 0])

# Check if missingness follows content_type
print("\nMissingness of search_volume grouped by content_type:")
print(df.groupby('content_type')['search_volume'].apply(lambda x: x.isnull().mean()))

Number of duplicate content_id rows: 0
Rows with avg_position == 0: 1205

Missing values per column (ratio):
search_volume        0.082267
competition          0.082267
competition_level    0.087000
cpc                  0.082267
main_intent          0.079133
word_count           0.256633
char_count           0.256633
provider_used        0.714600
model_used           0.191100
word_count_tier      0.256633
char_count_tier      0.256633
scroll_rate          0.004167
trend_pct            0.112933
dtype: float64

Missingness of search_volume grouped by content_type:
content_type
comparison article    0.000000
feedly article        1.000000
keyword article       0.013673
Name: search_volume, dtype: float64


## 4. Data limits

### Critical Limitations & Gotchas
1. **History depth differences**: The dataset is a snapshot. In the big warehouse release, different clients have varying history depths (`gsc_data_start`). A global time window will introduce bias or silent filtering.
2. **Zero-filling prior to tracking**: Rows before `ga4_data_start` are filled with zeros but marked with `ga4_data_available = FALSE`. Treating these as genuine "zero engagement" will severely mislead any recommendation model.
3. **The Label Trap**: Columns `trend_direction` and `trend_pct` directly compute `is_declining_label`. Including these as features guarantees $100\%$ accuracy in training but zero real-world value (pure data leakage).

In [18]:
# 1. Verify Label Trap
corr_with_trend = df.groupby('trend_direction')['trend_pct'].describe()
print("Trend percent characteristics by trend direction:")
print(corr_with_trend)

# 2. Check value distributions of rate metrics to confirm 100x scale
print("\nMax values of rate columns (checking if some exceed 100%):")
for col in ['ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']:
    print(f"{col}: max = {df[col].max():.2f}, mean = {df[col].mean():.2f}")

Trend percent characteristics by trend direction:
                   count        mean          std    min   25%    50%    75%  \
trend_direction                                                                
down             16262.0  -58.113830    23.488605 -100.0 -75.9 -55.60  -38.5   
flat                 0.0         NaN          NaN    NaN   NaN    NaN    NaN   
new                  0.0         NaN          NaN    NaN   NaN    NaN    NaN   
stable            5962.0   -3.185944    11.054723  -20.0 -12.7  -3.80    5.0   
up                4388.0  190.673997  1145.029477   20.0  36.5  62.55  123.1   

                     max  
trend_direction           
down               -20.0  
flat                 NaN  
new                  NaN  
stable              20.0  
up               44900.0  

Max values of rate columns (checking if some exceed 100%):
ctr: max = 100.00, mean = 0.51
engagement_rate: max = 100.00, mean = 2.53
scroll_rate: max = 300.00, mean = 18.21
ai_traffic_pct: max = 300.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.